In [1]:
%load_ext lab_black
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import polars as pl
import time
from tqdm.notebook import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.preprocessing import OrdinalEncoder
import os
import gc
import matplotlib.pyplot as plt
import warnings
import sys
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
import random

os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
from merlin.loader.torch import Loader
from merlin.io import Dataset as Dataset_merlin

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(device)

sys.path.append("../src/otto_recommender")
import utils

warnings.filterwarnings("ignore")
tqdm.pandas()

pd.set_option("max_rows", 500)

%matplotlib inline

cuda
INFO: Pandarallel will run on 16 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [2]:
def seed_everything(seed):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # set True to be faster
    # torch.use_deterministic_algorithms(True)
    print(f"Setting all seeds to be {seed} to reproduce...")


seed_everything(42)

Setting all seeds to be 42 to reproduce...


In [3]:
class Config:
    col_user = "user"
    col_aid = "aid"
    col_type = "type"
    col_n_session = "n_session"
    col_n_day = "n_day"
    col_n_week = "n_week"
    col_dayofweek = "dayofweek"
    col_hour = "hour"
    col_ts = "ts"
    col_num_action = "num_action"
    col_train_seq = "train_seq"
    col_target = "target"
    col_type_group = "type_group"
    path_data = "../data/02_intermediate/"
    batch_size_train = 512
    batch_size_infer = 512
    n_epochs = 10
    max_len = 10
    model_name = "def_tune_"
    hidden_dim = 128
    embedding_dim = 100
    init_lr = 0.0005
    cur_model_path = "def_tune__epoch11_larger_lr_and_bs.pth"
    cur_model_output_path = "../data/07_model_output/"


config = Config()

In [4]:
%%time

X_train, X_val, X_val_seq, ordinal_encoder = utils.get_train_val_dfs(config)
# X_train, X_val, X_val_seq, ordinal_encoder = utils.dfs_interactions_to_list(
#     config, 
#     X_train, 
#     X_val, 
#     X_val_seq, 
#     ordinal_encoder, 
#     return_only_val=True, 
#     truncate_val=True
# )
pad = ordinal_encoder.unknown_value

CPU times: user 10min 52s, sys: 1min 34s, total: 12min 27s
Wall time: 3min 56s


In [5]:
mod = "val"

In [6]:
%%time

dict_models_preds = dict()

cols_to_drop = ['index']

cols_to_rename = dict()
cols_to_rename[config.col_user] = 'session'
cols_to_rename['item_id'] = 'preds'

cols_to_add_model_name = ['preds']

model_names = ['rnn', 'cooc_smart', 'bm25', 'tfidf', 'cosine', 'last_items']

for model_name in model_names:
    dict_models_preds[model_name] = pl.read_parquet(config.cur_model_output_path + f'cur_{model_name}_df_{mod}_preds_100.parquet')
    for col in cols_to_drop:
        if col in dict_models_preds[model_name]:
            dict_models_preds[model_name] = dict_models_preds[model_name].drop(col)
    for col in cols_to_rename:
        if col in dict_models_preds[model_name]:
            dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: cols_to_rename[col]})
    for col in cols_to_add_model_name:
        if col in dict_models_preds[model_name]:
            dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: model_name + '_' + col})
    if model_name == 'rnn':
        dict_models_preds[model_name] = dict_models_preds[model_name].rename({'preds_scores': 'rnn_scores'})
        dict_models_preds[model_name] = dict_models_preds[model_name].with_column(
            pl.col("rnn_scores")
                .rank(method="ordinal", reverse=True)
                .over(["session"])
                .alias("rnn_rank")
        )
    elif model_name in ['cooc_smart']:
        for col in ['cooc_triplet_score_0', 'cooc_triplet_rank_0', 'cooc_score_0', 'cooc_rank_0']:
            if col in dict_models_preds[model_name]:
                dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: col[:-2]})
    for col in dict_models_preds[model_name].columns:
        if 'preds' in col:
            dict_models_preds[model_name] = dict_models_preds[model_name].rename({col: 'item_id'})

CPU times: user 42 s, sys: 5.19 s, total: 47.2 s
Wall time: 12.4 s


In [7]:
dict_models_preds["bm25"].head()

item_id,bm25_score,bm25_rank,session
i64,f64,i64,i64
641329,6743.168437,1,13267584
1568011,1707.0542,2,13267584
168118,1469.169167,3,13267584
131533,1236.32086,4,13267584
341426,1186.261317,5,13267584


In [8]:
dict_models_preds["last_items"].head()

item_id,last_items_rank,session
i64,i64,i64
466160,1,13145152
481038,1,14000256
636741,1,13492928
577304,2,13492928
1768724,1,13592640


In [9]:
model_names

['rnn', 'cooc_smart', 'bm25', 'tfidf', 'cosine', 'last_items']

In [10]:
%%time

df_merged = dict_models_preds[model_names[0]]
for i in tqdm(range(1, len(model_names))):
    model_name_cur = model_names[i]
    df_merged = utils.polars_outer_merge(df_merged, dict_models_preds[model_name_cur])

  0%|          | 0/5 [00:00<?, ?it/s]

CPU times: user 7min 31s, sys: 2min 7s, total: 9min 39s
Wall time: 4min 2s


In [11]:
%%time

X_val_lists = X_val.groupby([config.col_user]).agg(
    [
        pl.col(config.col_aid).list().alias(config.col_train_seq),
    ]
)

CPU times: user 2.32 s, sys: 472 ms, total: 2.79 s
Wall time: 226 ms


In [12]:
%%time

df_targets = utils.create_targets_df(list(X_val_lists.to_pandas()[config.col_train_seq]))

CPU times: user 4.19 s, sys: 292 ms, total: 4.48 s
Wall time: 4.43 s


In [13]:
%%time

df_users = pl.from_pandas(X_val_lists.to_pandas()[[config.col_user]].reset_index())
df_users = df_users.join(df_targets, on='index', how='inner')
df_users = df_users.drop('index')
df_users = df_users.rename({config.col_user: 'session'})

CPU times: user 2.45 s, sys: 640 ms, total: 3.09 s
Wall time: 583 ms


In [14]:
%%time

df_merged = utils.polars_outer_merge(df_merged, df_users)

CPU times: user 41.1 s, sys: 31.5 s, total: 1min 12s
Wall time: 34.3 s


In [15]:
del df_users, dict_models_preds
gc.collect()

162

In [16]:
columns_from_models = [
    col for col in df_merged.columns if "_scores" in col or "_rank" in col
]

In [17]:
%%time

df_merged = pl.from_pandas(df_merged.to_pandas().dropna(subset=columns_from_models, thresh=1))

CPU times: user 17 s, sys: 24.3 s, total: 41.3 s
Wall time: 21.3 s


In [18]:
df_merged

item_id,session,rnn_scores,rnn_rank,cooc_score,cooc_rank,bm25_score,bm25_rank,tfidf_score,tfidf_rank,cosine_score,cosine_rank,last_items_rank,target
i64,i64,f32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
1497089,13556096,0.002568,1.0,null,null,null,null,null,null,null,null,null,null
1712351,13556096,0.002473,2.0,null,null,null,null,null,null,null,null,null,null
1636724,13556096,0.001665,3.0,null,null,null,null,null,null,null,null,null,null
83986,13556096,0.001621,4.0,null,null,null,null,null,null,null,null,null,null
1461779,13556096,0.001564,5.0,null,null,null,null,null,null,null,null,null,null
654388,13556096,0.001355,6.0,null,null,null,null,null,null,null,null,null,null
539417,13556096,0.001254,7.0,null,null,null,null,null,null,null,null,null,null
1087291,13556096,0.001104,8.0,null,null,null,null,null,null,null,null,null,null
959208,13556096,0.001081,9.0,null,null,null,null,null,null,null,null,null,null


In [27]:
df_merged[columns_from_models].is_not_null()

AttributeError: 'DataFrame' object has no attribute 'is_not_null'

In [22]:
df_merged

item_id,session,rnn_scores,rnn_rank,cooc_score,cooc_rank,bm25_score,bm25_rank,tfidf_score,tfidf_rank,cosine_score,cosine_rank,last_items_rank,target
i64,i64,f32,u32,f64,i32,f64,i64,f64,i64,f64,i64,i64,f64
1497089,13556096,0.002568,1,null,null,null,null,null,null,null,null,null,null
1712351,13556096,0.002473,2,null,null,null,null,null,null,null,null,null,null
1636724,13556096,0.001665,3,null,null,null,null,null,null,null,null,null,null
83986,13556096,0.001621,4,null,null,null,null,null,null,null,null,null,null
1461779,13556096,0.001564,5,null,null,null,null,null,null,null,null,null,null
654388,13556096,0.001355,6,null,null,null,null,null,null,null,null,null,null
539417,13556096,0.001254,7,null,null,null,null,null,null,null,null,null,null
1087291,13556096,0.001104,8,null,null,null,null,null,null,null,null,null,null
959208,13556096,0.001081,9,null,null,null,null,null,null,null,null,null,null
